# generate & fuse Pokémon by name

Load the conditional DDPM trained by `pokemon_train.ipynb`, and do two things:
1. **Generate by name** a specified Pokémon;
2. **Fuse two**—— interpolate the embeddings of two names and then sample.

> This notebook is **self-contained**: the UNet and Diffusion definitions are inlined directly below (exactly the same as `pokemon_train.ipynb`), with no dependency on any `.py` file.

In [ ]:
import os, json, math, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

import os as _os
_os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# points to the output directory of a specific experiment (corresponds to EXP_NAME at training time)
EXP_NAME  = "exp05_pokemon20aug_bigUNet-CFG-attn-EMA"
OUT_DIR   = "."
CKPT      = os.path.join(OUT_DIR, "checkpoints", "ckpt_ep300.pt")   # ← change to the epoch you want
IMG_SIZE  = 96
TIMESTEPS = 1000
# Model capacity must match training (otherwise loading weights will error)
BASE      = 128
N_BLOCKS  = 2
print("device:", device, "| experiment:", EXP_NAME)

## Model and diffusion definitions (consistent with training, inlined here for standalone loading)

In [ ]:
def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, groups=8):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.GroupNorm(groups, in_ch), nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.cond_mlp = nn.Sequential(nn.SiLU(), nn.Linear(c_dim, out_ch))
        self.block2 = nn.Sequential(
            nn.GroupNorm(groups, out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, c):
        h = self.block1(x)
        h = h + self.cond_mlp(c)[:, :, None, None]
        h = self.block2(h)
        return h + self.skip(x)

class AttnBlock(nn.Module):
    """Spatial self-attention: treat each pixel as a token so it can see the whole image. With residual."""
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H * W).transpose(1, 2)
        h, _ = self.mha(h, h, h)
        h = h.transpose(1, 2).reshape(B, C, H, W)
        return x + h

class Stage(nn.Module):
    """One resolution level: n_blocks ResBlocks, with optional self-attention at the end."""
    def __init__(self, in_ch, out_ch, c_dim, n_blocks=2, attn=False):
        super().__init__()
        self.blocks = nn.ModuleList(
            [ResBlock(in_ch if i == 0 else out_ch, out_ch, c_dim) for i in range(n_blocks)])
        self.attn = AttnBlock(out_ch) if attn else None
    def forward(self, x, c):
        for b in self.blocks:
            x = b(x, c)
        if self.attn is not None:
            x = self.attn(x)
        return x

class UNet(nn.Module):
    def __init__(self, in_ch=3, base=128, c_dim=256, num_classes=None, n_blocks=2):
        super().__init__()
        self.c_dim = c_dim
        self.num_classes = num_classes
        self.time_mlp = nn.Sequential(
            nn.Linear(c_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))
        if num_classes is not None:
            self.label_emb = nn.Embedding(num_classes + 1, c_dim)   # +1 for the null class
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down1 = Stage(base,      base,     c_dim, n_blocks)
        self.down2 = Stage(base,      base * 2, c_dim, n_blocks)
        self.down3 = Stage(base * 2,  base * 4, c_dim, n_blocks, attn=True)   # 24 +attn
        self.downsample = nn.AvgPool2d(2)
        self.mid = Stage(base * 4, base * 4, c_dim, n_blocks, attn=True)      # 12 +attn
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = Stage(base * 4 + base * 4, base * 2, c_dim, n_blocks, attn=True)  # 24 +attn
        self.up2 = Stage(base * 2 + base * 2, base,     c_dim, n_blocks)
        self.up1 = Stage(base     + base,     base,     c_dim, n_blocks)
        self.out = nn.Sequential(
            nn.GroupNorm(8, base), nn.SiLU(),
            nn.Conv2d(base, in_ch, 3, padding=1))

    def cond(self, t, y=None, y_emb=None):
        c = self.time_mlp(sinusoidal_embedding(t, self.c_dim))
        if self.num_classes is not None:
            if y_emb is None:
                if y is None:
                    y = torch.full((t.size(0),), self.num_classes,
                                   device=t.device, dtype=torch.long)
                y_emb = self.label_emb(y)
            c = c + y_emb
        return c

    def forward(self, x, t, y=None, y_emb=None):
        c = self.cond(t, y, y_emb)
        x = self.in_conv(x)
        s1 = self.down1(x, c);  x = self.downsample(s1)
        s2 = self.down2(x, c);  x = self.downsample(s2)
        s3 = self.down3(x, c);  x = self.downsample(s3)
        x = self.mid(x, c)
        x = self.upsample(x); x = self.up3(torch.cat([x, s3], 1), c)
        x = self.upsample(x); x = self.up2(torch.cat([x, s2], 1), c)
        x = self.upsample(x); x = self.up1(torch.cat([x, s1], 1), c)
        return self.out(x)

In [ ]:
class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = timesteps
        self.device = device
        beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        alpha = 1.0 - beta
        self.beta = beta
        self.alpha = alpha
        self.alpha_bar = torch.cumprod(alpha, dim=0)

    def q_sample(self, x0, t, noise):
        ab = self.alpha_bar[t][:, None, None, None]
        return ab.sqrt() * x0 + (1.0 - ab).sqrt() * noise

    @torch.no_grad()
    def sample(self, model, n, y=None, y_emb=None, guidance=3.0, img_size=96, channels=3):
        model.eval()
        x = torch.randn(n, channels, img_size, img_size, device=self.device)
        use_cfg = (model.num_classes is not None) and (y is not None or y_emb is not None)
        if use_cfg:
            null = torch.full((n,), model.num_classes, device=self.device, dtype=torch.long)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            if use_cfg:
                eps_c = model(x, t, y=y, y_emb=y_emb)
                eps_u = model(x, t, y=null)
                pred = eps_u + guidance * (eps_c - eps_u)
            else:
                pred = model(x, t)
            beta_t, alpha_t, ab_t = self.beta[i], self.alpha[i], self.alpha_bar[i]
            mean = (1.0 / alpha_t.sqrt()) * (x - (beta_t / (1.0 - ab_t).sqrt()) * pred)
            x = mean + (beta_t.sqrt() * torch.randn_like(x) if i > 0 else 0.0)
        model.train()
        return x

## Load checkpoint

Read `classes.json` (the name list saved during training) and the weights, then build the model.

In [ ]:
classes = json.load(open(os.path.join(OUT_DIR, "classes.json"), encoding="utf-8"))
NUM_CLASSES = len(classes)
print("num classes:", NUM_CLASSES, "| e.g.:", classes[:3])

model = UNet(in_ch=3, base=BASE, num_classes=NUM_CLASSES, n_blocks=N_BLOCKS).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device))   # checkpoint is EMA weights
model.eval()
diff = Diffusion(timesteps=TIMESTEPS, device=device)
print("loaded:", CKPT)

## Utility: look up class index by name

The class names of the 20-Pokémon dataset are lowercase English names (e.g. `pikachu`, `charmander`), matched directly by keyword.

In [ ]:
def find_idx(query):
    """Return (index, name) for all names containing query."""
    q = query.lower()
    hits = [(i, n) for i, n in enumerate(classes) if q in n.lower()]
    if not hits:
        print("Not found:", query);
    return hits

def show(imgs, title=""):
    imgs = (imgs.clamp(-1, 1) + 1) / 2
    grid = make_grid(imgs.cpu(), nrow=min(len(imgs), 4)).permute(1, 2, 0).numpy()
    plt.figure(figsize=(8, 8)); plt.imshow(grid); plt.axis("off")
    plt.title(title); plt.show()

def emb_of(name):
    """Get the name embedding of a given Pokémon (1, c_dim)."""
    return model.label_emb(torch.tensor([find_idx(name)[0][0]], device=device))

def slerp(a, b, t):
    """Spherical interpolation: interpolate between two embeddings while [preserving vector length].
    Linear interpolation shortens the norm (0.5A+0.5B is shorter than both A and B) -> goes out of distribution -> generates all white;
    SLERP travels along the sphere with constant length, so fusion works correctly. t=0 is a, t=1 is b. Returns (1, c_dim)."""
    a, b = a.squeeze(0), b.squeeze(0)
    an, bn = a / a.norm(), b / b.norm()
    om = torch.acos((an * bn).sum().clamp(-1, 1))     # angle between the two vectors
    so = torch.sin(om)
    if so < 1e-6:                                      # near-parallel, degenerates to linear
        out = (1 - t) * a + t * b
    else:
        out = (torch.sin((1 - t) * om) / so) * a + (torch.sin(t * om) / so) * b
    return out.unsqueeze(0)

find_idx("pikachu")[:5]

## 1. Generate by name

Pick a Pokémon and generate N images. Larger `guidance` fits the name more closely (too large may distort; 3~6 is common).

In [ ]:
NAME = "pikachu"      # ← change to the Pokémon you want
N = 8
GUIDANCE = 4.0

idx = find_idx(NAME)[0][0]
y = torch.full((N,), idx, device=device, dtype=torch.long)
imgs = diff.sample(model, N, y=y, guidance=GUIDANCE, img_size=IMG_SIZE)
show(imgs, f"{classes[idx]}  x{N}")

## 2. 🧬 Fuse two Pokémon

Interpolate the embeddings of two names, then sample with the fused embedding. **Key: use SLERP (spherical interpolation) rather than plain linear interpolation.**

Linear interpolation `0.5*A + 0.5*B` **shortens the vector norm** (averaging makes it shorter), landing in an "out-of-distribution" region the model has never seen; combined with high CFG guidance this pushes generation to **all-white saturation** (we hit exactly this pitfall). **SLERP travels along the sphere, keeping length unchanged**, so fusion works correctly:

```python
fused = slerp(emb_A, emb_B, alpha)   # alpha=0 pure A, 1 pure B, in between is the fusion
```

In [ ]:
NAME_A = "charmander"   # ← Pokémon A
NAME_B = "squirtle"     # ← Pokémon B
ALPHA  = 0.5            # fusion ratio (0=pure A, 1=pure B)
N = 8
GUIDANCE = 4.0

emb_a = emb_of(NAME_A)
emb_b = emb_of(NAME_B)
fused = slerp(emb_a, emb_b, ALPHA).repeat(N, 1)   # spherical interpolation, norm-preserving -> (N, c_dim)

imgs = diff.sample(model, N, y_emb=fused, guidance=GUIDANCE, img_size=IMG_SIZE)
show(imgs, f"{NAME_A} x {NAME_B}  (alpha={ALPHA})")

## 3. View the "gradient process" of fusion

Sweep alpha from 0 to 1 to see how one Pokémon gradually morphs into another—a very intuitive demonstration
what embedding interpolation is doing.

In [ ]:
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
emb_a = emb_of(NAME_A)
emb_b = emb_of(NAME_B)

imgs = []
for a in alphas:
    fused = slerp(emb_a, emb_b, a)              # spherical interpolation, preserves the norm at every alpha
    imgs.append(diff.sample(model, 1, y_emb=fused, guidance=GUIDANCE, img_size=IMG_SIZE)[0])
show(torch.stack(imgs), f"{NAME_A} → {NAME_B}   alpha = {alphas}")

## 4. Another fusion: compositional CFG (clearer structure)

Instead of feeding a "middle embedding", we **guide separately with the two real embeddings A and B, then mix their guidance directions by alpha**:

$$\varepsilon = \varepsilon_{\text{null}} + g\big[(1-\alpha)(\varepsilon_A-\varepsilon_{\text{null}}) + \alpha(\varepsilon_B-\varepsilon_{\text{null}})\big]$$

Upside: never feed out-of-distribution intermediate vectors, so **intermediate structure is usually clearer than SLERP**. Downside: transitions are more "either-or", and intermediate colors may turn grayish. Each has its own flavor, compare and play. If the middle turns gray, drop `GUIDANCE` to 2~3.

In [ ]:
NAME_A = "charmander"   # ← Pokémon A
NAME_B = "squirtle"     # ← Pokémon B
ALPHA  = 0.5            # mix ratio (0=pure A, 1=pure B)
N = 8
GUIDANCE = 4.0

@torch.no_grad()
def sample_compose(ia, ib, alpha, n, guidance, img_size=96):
    """Compositional CFG: two real conditions A and B each guide, mixing the guidance directions by alpha."""
    x = torch.randn(n, 3, img_size, img_size, device=device)
    yA   = torch.full((n,), ia, device=device, dtype=torch.long)
    yB   = torch.full((n,), ib, device=device, dtype=torch.long)
    null = torch.full((n,), model.num_classes, device=device, dtype=torch.long)
    for i in reversed(range(diff.T)):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        eu = model(x, t, y=null)
        eA = model(x, t, y=yA)
        eB = model(x, t, y=yB)
        pred = eu + guidance * ((1 - alpha) * (eA - eu) + alpha * (eB - eu))
        bt, at, abt = diff.beta[i], diff.alpha[i], diff.alpha_bar[i]
        mean = (1 / at.sqrt()) * (x - (bt / (1 - abt).sqrt()) * pred)
        x = mean + (bt.sqrt() * torch.randn_like(x) if i > 0 else 0.0)
    return x

ia = find_idx(NAME_A)[0][0]
ib = find_idx(NAME_B)[0][0]
imgs = sample_compose(ia, ib, ALPHA, N, GUIDANCE, IMG_SIZE)
show(imgs, f"{NAME_A} + {NAME_B}  compose (alpha={ALPHA})")

## Notes

- **Fusion must use SLERP** (already used above). Linear interpolation shortens the norm → out of distribution → all white, this is a pitfall actually hit.
- If fusion is still whitish/distorted, lower `GUIDANCE` to 2~3 (high guidance amplifies out-of-distribution instability).
- Current exp05 (large data + large model) already generates well by name; fusion is a mix of color/outline/features, which is normal and expected.

## 5. Unconditional generation

Without a name, the model samples with the null condition → "a generic Pokémon-like creature" (no specific one).
This works because training applied CFG condition dropout, so the model has learned null.

In [ ]:
# unconditional generation: don't pass y / y_emb, the model automatically uses the null condition
imgs = diff.sample(model, 8, img_size=IMG_SIZE)
show(imgs, "unconditional generation (no name specified)")

## 6. Fuse two images (noise-space interpolation, no image encoder needed)

Our model **has no image encoder**, so it can't turn an arbitrary image into a conditioning embedding. But we can fuse two images in **noise space**:
Noise each of the two images up to an intermediate step `t_start` → spherically interpolate in the noise → then denoise back (using null unconditional).

- The larger `t_start` (e.g. 800) → the more abstract, more like a new creature made from nothing; the smaller (e.g. 300) → the closer to overlaying the two original images
- Because the model has only learned Pokémon, any image gets "Pokémon-ized" too